# *DistilRoBERTa*


In [ ]:
import numpy as np
import pandas as pd
import os, copy
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import (
    RobertaTokenizer, RobertaForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import f1_score, classification_report
from sklearn.utils.class_weight import compute_class_weight

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

LABEL2ID  = {'google': 0, 'anthropic': 1, 'meta': 2, 'openai': 3, 'human': 4}
ID2LABEL  = {v: k for k, v in LABEL2ID.items()}
N_CLASSES = 5

In [ ]:
df_full  = pd.read_csv('../datasets/dataset_v2_full.csv',       sep=';')
df_ex    = pd.read_csv('../datasets/dataset-exemplos.csv',      sep=';')
df_subm1 = pd.read_csv('../Subm1/subm1_labels_revealed.csv',    sep=';')

def load_xy(df):
    texts  = df['Text'].fillna('').tolist()
    labels = [LABEL2ID[l.lower()] for l in df['Label'].tolist()]
    return texts, labels

texts_synth, y_synth = load_xy(df_full)
texts_real,  y_real  = load_xy(df_subm1)
texts_val,   y_val   = load_xy(df_ex)

REAL_WEIGHT = 10
texts_train = texts_synth + texts_real * REAL_WEIGHT
y_train     = y_synth     + y_real     * REAL_WEIGHT

cw = compute_class_weight('balanced', classes=np.arange(N_CLASSES), y=y_real + y_val)
class_weights = torch.tensor(cw, dtype=torch.float32).to(device)

print(f'treino : {len(texts_train)} ({len(texts_synth)} sintéticos + {len(texts_real)*REAL_WEIGHT} reais×{REAL_WEIGHT})')
print(f'validação    : {len(texts_val)} reais')
print('class weights:', {ID2LABEL[i]: f'{w:.2f}' for i, w in enumerate(cw)})

In [ ]:
MODEL_NAME = 'distilroberta-base'
MAX_LEN    = 128

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.encodings = tokenizer(
            texts, truncation=True, padding='max_length',
            max_length=max_len, return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self): return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids':      self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'labels':         self.labels[idx]
        }

def evaluate(model, loader):
    model.eval()
    preds_all, true_all = [], []
    with torch.no_grad():
        for batch in loader:
            ids  = batch['input_ids'].to(device)
            mask = batch['attention_mask'].to(device)
            out  = model(input_ids=ids, attention_mask=mask)
            preds_all.extend(torch.argmax(out.logits, dim=1).cpu().tolist())
            true_all.extend(batch['labels'].tolist())
    acc = sum(p == t for p, t in zip(preds_all, true_all)) / len(true_all)
    f1  = f1_score(true_all, preds_all, average='macro')
    return acc, f1, preds_all

def label_smoothing_loss(logits, labels, n_classes, smoothing=0.1, weights=None):
    log_probs = F.log_softmax(logits, dim=-1)
    smooth_target = torch.full_like(log_probs, smoothing / (n_classes - 1))
    smooth_target.scatter_(1, labels.unsqueeze(1), 1.0 - smoothing)
    loss = -(smooth_target * log_probs).sum(dim=-1)
    if weights is not None:
        loss = loss * weights[labels]
    return loss.mean()

print(f'modelo: {MODEL_NAME}')

In [ ]:
# ── hiperparâmetros (melhor configuração da search) ────────────────────────
LR           = 3e-5
LABEL_SMOOTH = 0.1
BATCH_SIZE   = 32
MAX_EPOCHS   = 15
PATIENCE     = 3
WARMUP_FRAC  = 0.1

print('a tokenizar...')
train_ds = TextDataset(texts_train, y_train, tokenizer, MAX_LEN)
val_ds   = TextDataset(texts_val,   y_val,   tokenizer, MAX_LEN)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds,   batch_size=64)
print(f'tokenização concluída!\n train batches: {len(train_loader)}')

In [ ]:
model = RobertaForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=N_CLASSES,
    id2label=ID2LABEL, label2id=LABEL2ID
).to(device)

optimizer    = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps  = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_FRAC)
scheduler    = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)

scaler = torch.cuda.amp.GradScaler(enabled=device.type == 'cuda')

best_f1    = 0.0
best_acc   = 0.0
best_state = None
no_improve = 0

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    total_loss = 0
    for batch in train_loader:
        ids    = batch['input_ids'].to(device)
        mask   = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=device.type == 'cuda'):
            logits = model(input_ids=ids, attention_mask=mask).logits
            loss   = label_smoothing_loss(logits, labels, N_CLASSES,
                                          smoothing=LABEL_SMOOTH, weights=class_weights)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    val_acc, val_f1, _ = evaluate(model, val_loader)
    marker = ' *' if val_f1 > best_f1 else ''
    print(f'Epoch {epoch:02d}/{MAX_EPOCHS} | loss={total_loss/len(train_loader):.4f} '
          f'| val_acc={val_acc:.4f} | val_f1={val_f1:.4f}{marker}')

    if val_f1 > best_f1:
        best_f1    = val_f1
        best_acc   = val_acc
        best_state = copy.deepcopy(model.state_dict())
        no_improve = 0
    else:
        no_improve += 1
        if no_improve >= PATIENCE:
            print(f'Early stopping at epoch {epoch}')
            break

model.load_state_dict(best_state)

os.makedirs('../models/model_bert', exist_ok=True)
model.save_pretrained('../models/model_bert')
tokenizer.save_pretrained('../models/model_bert')
print('modelo guardado em ../models/model_bert')
print(f'\nmelhor modelo: val_acc={best_acc:.4f}  val_f1={best_f1:.4f}')

## *avaliação com dataset-exemplos*

In [ ]:
val_acc, val_f1, val_preds = evaluate(model, val_loader)
print(f'[dataset-exemplos] accuracy={val_acc:.4f}  f1-macro={val_f1:.4f}')
print()
print(classification_report(
    [ID2LABEL[l] for l in y_val],
    [ID2LABEL[p] for p in val_preds],
    digits=3
))